# Two-group generalization decoding: is the tone code talker-invariant?

Group-level analysis of FLT2's `decoding_generalization.py`: learners (non-Mandarin) vs. native listeners (Mandarin).

- **Tone across talkers:** a linear classifier trained to label tone on three talkers, tested on the held-out fourth (four folds, averaged). Above chance means a tone code that transfers to a new voice, i.e. a category rather than the acoustics of particular stimuli. RSA's tone fit can't separate those.
- **Talker across tones:** the same with roles swapped. A control: talker identity isn't learned, so it shouldn't follow the learning pattern.

The folds split stimuli, not runs, so learning between runs doesn't count against the classifier. The `_held-out-run` versions also train on one run and test on the other: more conservative, but lowered by learning between runs (as in leave-one-run-out decoding), so they're a sensitivity check, mainly informative for natives.

Values are accuracy − chance (chance 0.25). Talker-specific patterns push leave-one-talker-out accuracy slightly *below* chance (seen in simulation), so above-chance tests are conservative where stimulus-specific signal is strong.

Predictions: natives generalize from the early stage. Learners generalize only later (positive trend), so the groups differ early and converge.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join('..', '06_univariate'))
import group_stats as gs
import multivariate_data as mvd

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 20)

bidsroot = '/ix1/bchandrasekaran/krs228/data/FLT/data_denoised'
dec_dir = os.path.join(mvd.DERIV_DIR, mvd.GLMSINGLE_DIRNAME, 'decoding_generalization')
out_dir = os.path.join(mvd.DERIV_DIR, mvd.GLMSINGLE_DIRNAME, 'stats_two-group', 'decoding_generalization')
os.makedirs(out_dir, exist_ok=True)

ANALYSES = ['tone_across_talker', 'talker_across_tone']
n_perms, n_boot, seed = 10000, 10000, 0   # matching robustness_checks.py
colors = {'non-Mandarin': '#2a6fdb', 'Mandarin': '#d9822b'}   # as in group_level_two_group.ipynb
labels = {'non-Mandarin': 'learners', 'Mandarin': 'natives'}

## 1. Load and check

In [ ]:
participants = mvd.load_participants(bidsroot)
dec = mvd.add_composites(mvd.load_decoding(dec_dir, participants))
print('no decoding output:', sorted(set(participants.participant_id) - set(dec.participant_id)))
print('ROIs:', sorted(dec.region_hemi.unique()))
gs.coverage(dec)

## 2. Accuracy by stage and group

Hemispheres averaged per participant; mean ± SEM, accuracy − chance.

In [ ]:
bilat = gs.bilateral_mean(dec)
regions = list(mvd.COMPOSITES) + sorted(r for r in bilat.region.unique() if r not in mvd.COMPOSITES)
for analysis in ANALYSES:
    sub = bilat[bilat.contrast == analysis]
    fig, axes = plt.subplots(1, len(regions), figsize=(2.2 * len(regions), 2.6), sharey=True)
    for ax, region in zip(np.atleast_1d(axes), regions):
        for group in gs.GROUPS:
            g = sub[(sub.region == region) & (sub.participant_group == group)].groupby('learning_stage').beta
            ax.errorbar(range(3), g.mean().reindex(gs.STAGE_ORDER), yerr=g.sem().reindex(gs.STAGE_ORDER),
                        marker='o', capsize=3, color=colors[group], label=labels[group])
        ax.axhline(0, color='0.6', lw=0.8)
        ax.set_xticks(range(3), gs.STAGE_ORDER)
        ax.set_title(region)
    np.atleast_1d(axes)[0].set_ylabel(f'{analysis.replace("_", " ")}\naccuracy − chance')
    np.atleast_1d(axes)[0].legend(frameon=False)
    fig.tight_layout()

## 3. Above chance within each group

One-sample t on accuracy − chance, per stage and stages averaged; FDR across regions within group × analysis × network (× stage).

In [ ]:
above = gs.baseline_tests(dec)
above_by_stage = gs.baseline_tests(dec, by_stage=True)
above.to_csv(os.path.join(out_dir, 'above-chance.tsv'), sep='\t', index=False)
above_by_stage.to_csv(os.path.join(out_dir, 'above-chance_by-stage.tsv'), sep='\t', index=False)
above_by_stage[above_by_stage.contrast.isin(ANALYSES)].pivot_table(
    index=['contrast', 'network', 'region'], columns=['participant_group', 'learning_stage'], values='t').round(2)

## 4. Does a talker-invariant tone code emerge with learning? Trend, learners vs. natives

Full battery on the composites, aCAU and aPUT; Welch/permutation with FDR for every region.

In [ ]:
trends = gs.subject_trends(dec)
focus = [r for r in list(mvd.COMPOSITES) + ['aCAU', 'aPUT'] if r in set(trends.region)]
battery = pd.concat([gs.trend_group_difference(trends, analysis, region, n_perms=n_perms, n_boot=n_boot, seed=seed)
                     for analysis in ANALYSES for region in focus], ignore_index=True)
battery.to_csv(os.path.join(out_dir, 'trend_focus-regions.tsv'), sep='\t', index=False)
per_region = dec[~dec.region.isin(mvd.COMPOSITES)]
trend_diff = gs.trend_difference_tests(gs.subject_trends(per_region), contrasts=ANALYSES, n_perms=n_perms, seed=seed)
trend_diff.to_csv(os.path.join(out_dir, 'trend_group-difference.tsv'), sep='\t', index=False)
battery.round(3)

## 5. Natives ahead early? Group difference per stage

In [ ]:
stage_diff = gs.group_difference_tests(per_region, contrasts=ANALYSES, by_stage=True, n_perms=n_perms, seed=seed)
stage_diff.to_csv(os.path.join(out_dir, 'group-difference_by-stage.tsv'), sep='\t', index=False)
stage_diff.pivot_table(index=['contrast', 'network', 'region'], columns='learning_stage', values='d')[list(gs.STAGE_ORDER)].round(2)

## 6. Sensitivity: held-out run

Only if FLT2 ran with `--hold_out_run`. For natives, accuracy should hold up when the test run differs; a large drop points to run-specific (non-generalizing) structure. For learners a drop is expected early.

In [ ]:
held = [c for c in dec.contrast.unique() if c.endswith('_held-out-run')]
if held:
    both = gs.bilateral_mean(dec[dec.contrast.isin(ANALYSES + held)])
    both['scheme'] = np.where(both.contrast.str.endswith('_held-out-run'), 'held-out-run', 'same-runs')
    both['analysis'] = both.contrast.str.replace('_held-out-run', '')
    print(both.pivot_table(index=['analysis', 'participant_group', 'learning_stage'], columns='scheme',
                           values='beta', aggfunc='mean').round(3))
else:
    print('no held-out-run results (rerun FLT2 decoding with --hold_out_run)')

## 7. Export

In [ ]:
dec.to_csv(os.path.join(out_dir, 'decoding.tsv'), sep='\t', index=False)
trends.to_csv(os.path.join(out_dir, 'trend_scalars.tsv'), sep='\t', index=False)
print('wrote', out_dir)